# WavLM Supervised Contrastive Fine-tuning on AbjadKids

Original Colab notebook used to fine-tune WavLM-base-plus with Supervised Contrastive Learning on the AbjadKids Arabic-children speech corpus.

This notebook is part of the HAYATY developmental-screening project. It reproduces the embedding-backbone selection experiment described in Section 5.1.4.1 of the thesis.

Validation is monitored every 100 training steps; the best-validation checkpoint is saved separately.

In [ ]:
# Mount Google Drive so checkpoints can be persisted across Colab sessions
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Install the core dependencies needed throughout the notebook:
# - datasets, huggingface_hub, hf_transfer: downloading AbjadKids from Hugging Face
# - accelerate, transformers: WavLM model and feature extractor
# - torchaudio, librosa, soundfile: audio loading and resampling
!pip install -q datasets huggingface_hub hf_transfer accelerate transformers torchaudio librosa soundfile

In [ ]:
import os

# Speed up model and dataset downloads from Hugging Face Hub
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

# Cache the downloaded models / datasets in /content (volatile but fast)
os.environ["HF_HOME"] = "/content/hf_cache"
os.environ["HF_DATASETS_CACHE"] = "/content/hf_datasets_cache"

In [ ]:
# Authenticate with the Hugging Face Hub.
# When prompted, paste a personal access token (read scope is sufficient).
from huggingface_hub import login
login()

In [ ]:
# Download the AbjadKids dataset locally so audio files can be read by soundfile/librosa
from huggingface_hub import snapshot_download

LOCAL_DATASET_DIR = "/content/AbjadKids"

snapshot_path = snapshot_download(
    repo_id="Aziz-snoubra/Abjad-Kids",
    repo_type="dataset",
    local_dir=LOCAL_DATASET_DIR,
    local_dir_use_symlinks=False,   # avoid symlinks so the files are physically copied
    resume_download=True,           # continue partial downloads if the session was interrupted
    max_workers=16
)

print("Downloaded to:", snapshot_path)

In [ ]:
# Sanity check: confirm the dataset folder exists and inspect its top-level layout.
# AbjadKids is organised into three top-level categories: alphabet / colors / numbers.
DATA_DIR = "/content/AbjadKids"

import os

print(os.path.exists(DATA_DIR))
print(os.listdir(DATA_DIR))

In [ ]:
# Walk the AbjadKids directory tree and build a flat dataframe of audio files.
# Each row records: file path, top-level category, and word-level label name.
import os
import pandas as pd
from pathlib import Path

audio_exts = [".wav", ".mp3", ".flac", ".ogg", ".m4a"]

rows = []

for category in ["alphabet", "colors", "numbers"]:
    category_dir = Path(DATA_DIR) / category

    # Each subfolder under the category corresponds to one word class
    for class_dir in category_dir.iterdir():
        if class_dir.is_dir():
            label_name = class_dir.name

            # Recursively collect every audio file under that word folder
            for file in class_dir.rglob("*"):
                if file.suffix.lower() in audio_exts:
                    rows.append({
                        "path": str(file),
                        "category": category,
                        "label_name": label_name
                    })

df = pd.DataFrame(rows)

print("Total files:", len(df))
print("Total labels:", df["label_name"].nunique())
df.head()

In [ ]:
# Inspect the class distribution to detect under-represented classes
# that may need to be excluded before training.
counts = df["label_name"].value_counts()

print("Number of classes:", counts.shape[0])
print("Min samples per class:", counts.min())
print("Max samples per class:", counts.max())
print("Mean samples per class:", counts.mean())

counts.head(20)

In [ ]:
# Build a deterministic label-to-id mapping (alphabetical order),
# then attach an integer label column for use by the model.
label_names = sorted(df["label_name"].unique())
label2id = {name: i for i, name in enumerate(label_names)}
id2label = {i: name for name, i in label2id.items()}

df["label"] = df["label_name"].map(label2id)

print("Classes:", len(label_names))
print(label_names[:20])
df.head()

In [ ]:
# Extract a per-file speaker identifier from the filename.
# AbjadKids filenames follow the pattern: Label_speaker_uuid.ext
# A reliable speaker id is required so train/val/test splits stay speaker-disjoint.
from pathlib import Path
import re

def extract_speaker(path):
    stem = Path(path).stem
    parts = stem.split("_")

    # Expected layout: Label_speaker_uuid -> the speaker is parts[1]
    if len(parts) >= 3:
        return parts[1].strip().lower()

    # Fallback for shorter filename patterns
    if len(parts) >= 2:
        return parts[1].strip().lower()

    return "unknown"

df["speaker"] = df["path"].apply(extract_speaker)

print("Total files:", len(df))
print("Labels:", df["label_name"].nunique())
print("Speakers:", df["speaker"].nunique())

df[["label_name", "speaker", "path"]].head(20)

In [ ]:
# Build the *initial* speaker-disjoint train/val/test split.
# This block is run on the un-cleaned dataframe so the next cells can verify
# that speakers do not leak across splits before the under-represented class
# is removed and the splits are rebuilt.
from sklearn.model_selection import train_test_split

speakers = df["speaker"].unique()

# 70% of speakers go to train, the remaining 30% go to val + test
train_speakers, temp_speakers = train_test_split(
    speakers,
    test_size=0.30,
    random_state=42
)

# The remaining 30% is split evenly into val and test
val_speakers, test_speakers = train_test_split(
    temp_speakers,
    test_size=0.50,
    random_state=42
)

# Materialise the dataframes for each split
train_df = df[df["speaker"].isin(train_speakers)].reset_index(drop=True)
val_df   = df[df["speaker"].isin(val_speakers)].reset_index(drop=True)
test_df  = df[df["speaker"].isin(test_speakers)].reset_index(drop=True)

print("Train:", len(train_df), "speakers:", train_df["speaker"].nunique(), "labels:", train_df["label_name"].nunique())
print("Val:", len(val_df), "speakers:", val_df["speaker"].nunique(), "labels:", val_df["label_name"].nunique())
print("Test:", len(test_df), "speakers:", test_df["speaker"].nunique(), "labels:", test_df["label_name"].nunique())

# Quick smoke test: this should always be empty
print("Speaker overlap train/test:", set(train_df["speaker"]) & set(test_df["speaker"]))

In [ ]:
# Sanity checks: confirm splits are speaker-disjoint and file-disjoint,
# and report any classes missing from val or test.
train_s = set(train_df["speaker"])
val_s   = set(val_df["speaker"])
test_s  = set(test_df["speaker"])

# All three intersections must be empty for a valid speaker-disjoint setup
print("Train-Val :", train_s & val_s)
print("Train-Test:", train_s & test_s)
print("Val-Test  :", val_s & test_s)

# Cross-check: speaker-disjoint should imply file-disjoint, but verify directly
train_p = set(train_df["path"])
val_p   = set(val_df["path"])
test_p  = set(test_df["path"])

print("File overlap Train/Val :", len(train_p & val_p))
print("File overlap Train/Test:", len(train_p & test_p))
print("File overlap Val/Test  :", len(val_p & test_p))

# Some classes can disappear from val or test due to the speaker split
# (e.g. a class for which all speakers ended up in train)
all_labels = set(df["label_name"])

missing_val = all_labels - set(val_df["label_name"])
missing_test = all_labels - set(test_df["label_name"])

print("Missing labels in Val:", missing_val)
print("Missing labels in Test:", missing_test)

In [ ]:
# The 'Walad' class is severely under-represented in AbjadKids and would
# destabilise both training and per-class evaluation, so it is excluded
# before the final splits are rebuilt.
df_clean = df[df["label_name"] != "Walad"].reset_index(drop=True)

print("Before:", df["label_name"].nunique(), len(df))
print("After:", df_clean["label_name"].nunique(), len(df_clean))

In [ ]:
# Rebuild the splits on the cleaned dataframe.
# This block is the *authoritative* split used by the rest of the notebook.
from sklearn.model_selection import train_test_split

df = df_clean.copy()

# Re-index the labels so the integer ids form a contiguous range starting at 0
label_names = sorted(df["label_name"].unique())
label2id = {name: i for i, name in enumerate(label_names)}
id2label = {i: name for name, i in label2id.items()}

df["label"] = df["label_name"].map(label2id)

speakers = df["speaker"].unique()

# 70% / 15% / 15% speaker-disjoint split with a fixed seed for reproducibility
train_speakers, temp_speakers = train_test_split(
    speakers,
    test_size=0.30,
    random_state=42
)

val_speakers, test_speakers = train_test_split(
    temp_speakers,
    test_size=0.50,
    random_state=42
)

train_df = df[df["speaker"].isin(train_speakers)].reset_index(drop=True)
val_df   = df[df["speaker"].isin(val_speakers)].reset_index(drop=True)
test_df  = df[df["speaker"].isin(test_speakers)].reset_index(drop=True)

print("Train:", len(train_df), "files | speakers:", train_df["speaker"].nunique(), "| labels:", train_df["label_name"].nunique())
print("Val:", len(val_df), "files | speakers:", val_df["speaker"].nunique(), "| labels:", val_df["label_name"].nunique())
print("Test:", len(test_df), "files | speakers:", test_df["speaker"].nunique(), "| labels:", test_df["label_name"].nunique())

# Track classes that may still be missing from val/test after the speaker split
all_labels = set(df["label_name"])
print("Missing labels in Val:", all_labels - set(val_df["label_name"]))
print("Missing labels in Test:", all_labels - set(test_df["label_name"]))

In [ ]:
# Re-install / pin the smaller subset of dependencies actually used by the
# training code to make the runtime self-consistent if the kernel restarts.
!pip install -q transformers torchaudio soundfile librosa tqdm

In [ ]:
# Imports for the training pipeline.
# Grouped by domain: stdlib, scientific, PyTorch, Hugging Face, tqdm.
import os, random, re, glob
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import soundfile as sf
import librosa

from torch.utils.data import Dataset, DataLoader, Sampler
from sklearn.model_selection import train_test_split
from transformers import Wav2Vec2FeatureExtractor, WavLMModel
from tqdm.auto import tqdm

# Use GPU when available; falls back to CPU on machines without CUDA
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE)

In [ ]:
# Audio dataset: loads, resamples, peak-normalises, and either crops or
# pads each clip to a fixed length so a batch of clips can be stacked.
SR = 16000           # WavLM expects 16 kHz mono audio
MAX_SECONDS = 2.5    # most AbjadKids clips fit within 2.5 s
MAX_LEN = int(SR * MAX_SECONDS)

class AbjadAudioDataset(Dataset):
    def __init__(self, dataframe, sr=16000, max_len=40000):
        self.df = dataframe.reset_index(drop=True)
        self.sr = sr
        self.max_len = max_len

    def __len__(self):
        return len(self.df)

    def load_audio(self, path):
        # Read the raw waveform (could be mono or stereo)
        wav, sr = sf.read(path, dtype="float32")

        # Mix down stereo -> mono so the model sees a single channel
        if wav.ndim > 1:
            wav = wav.mean(axis=1)

        # Resample to the model's expected rate (16 kHz) only when needed
        if sr != self.sr:
            wav = librosa.resample(wav, orig_sr=sr, target_sr=self.sr)

        wav = wav.astype(np.float32)

        # Peak-normalise to keep amplitude consistent across recordings
        peak = np.max(np.abs(wav)) + 1e-8
        wav = wav / peak

        # Random crop if the clip is longer than max_len, else right-pad with zeros
        if len(wav) > self.max_len:
            start = random.randint(0, len(wav) - self.max_len)
            wav = wav[start:start+self.max_len]
        else:
            pad = self.max_len - len(wav)
            wav = np.pad(wav, (0, pad))

        return wav

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        wav = self.load_audio(row["path"])
        label = int(row["label"])

        return {
            "input_values": torch.tensor(wav, dtype=torch.float32),
            "label": torch.tensor(label, dtype=torch.long),
            "label_name": row["label_name"]
        }

# Wrap each split in an AbjadAudioDataset instance
train_dataset = AbjadAudioDataset(train_df)
val_dataset   = AbjadAudioDataset(val_df)
test_dataset  = AbjadAudioDataset(test_df)

print(len(train_dataset), len(val_dataset), len(test_dataset))

In [ ]:
# Balanced batch sampler:
# Supervised contrastive learning (SupCon) requires *at least two* samples
# per class in every batch so that positive pairs always exist. A vanilla
# random sampler does not guarantee this, hence the custom sampler below.
class BalancedBatchSampler(Sampler):
    def __init__(self, dataframe, n_classes=8, n_samples=4):
        self.df = dataframe.reset_index(drop=True)
        self.n_classes = n_classes      # how many distinct classes per batch
        self.n_samples = n_samples      # how many samples per chosen class

        # Pre-compute, for each label, the list of dataset indices
        self.label_to_indices = defaultdict(list)
        for idx, label in enumerate(self.df["label"]):
            self.label_to_indices[int(label)].append(idx)

        self.labels = list(self.label_to_indices.keys())
        self.batch_size = n_classes * n_samples   # 8 x 4 = 32 by default

    def __iter__(self):
        # Infinite stream of balanced batches
        while True:
            # Pick n_classes distinct labels uniformly at random
            selected_labels = random.sample(self.labels, self.n_classes)
            batch = []

            # For each selected label, pick n_samples examples (with replacement)
            for label in selected_labels:
                indices = self.label_to_indices[label]
                chosen = random.choices(indices, k=self.n_samples)
                batch.extend(chosen)

            # Shuffle within the batch so the model does not see grouped labels
            random.shuffle(batch)
            yield batch

    def __len__(self):
        # Approximate epoch length in batches
        return len(self.df) // self.batch_size

# Stack waveforms and labels from a list of dataset items into batched tensors
def collate_fn(batch):
    input_values = torch.stack([x["input_values"] for x in batch])
    labels = torch.stack([x["label"] for x in batch])
    return {
        "input_values": input_values,
        "labels": labels
    }

# Train DataLoader: balanced batches, no extra worker processes
train_sampler = BalancedBatchSampler(train_df, n_classes=8, n_samples=4)

train_loader = DataLoader(
    train_dataset,
    batch_sampler=train_sampler,
    collate_fn=collate_fn,
    num_workers=0,
    pin_memory=True
)

# Validation DataLoader: same balanced-batch shape so SupCon loss is well-defined
val_sampler = BalancedBatchSampler(val_df, n_classes=8, n_samples=4)

val_loader = DataLoader(
    val_dataset,
    batch_sampler=val_sampler,
    collate_fn=collate_fn,
    num_workers=0,
    pin_memory=True
)

# Smoke-test the train loader: shapes should be [32, MAX_LEN] and [32]
batch = next(iter(train_loader))
print("Train batch input shape:", batch["input_values"].shape)
print("Train batch labels shape:", batch["labels"].shape)
print(torch.unique(batch["labels"], return_counts=True))

In [ ]:
# Model: WavLM-base-plus encoder followed by a 2-layer projection head.
# The projection output is L2-normalised so that the SupCon loss reduces to
# a cosine-similarity formulation on the unit hypersphere.
MODEL_NAME = "microsoft/wavlm-base-plus"

# The feature extractor is loaded for completeness; the WavLM model used
# below works directly on raw float32 waveforms so its output is the only
# tensor we read from this cell.
feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained(MODEL_NAME)

class WavLMSupConModel(nn.Module):
    def __init__(self, model_name=MODEL_NAME, projection_dim=128):
        super().__init__()

        # Backbone: pretrained WavLM-base-plus encoder (12 transformer layers, 768-dim)
        self.wavlm = WavLMModel.from_pretrained(model_name)
        hidden_size = self.wavlm.config.hidden_size

        # Projection head: 768 -> 256 -> 128, with ReLU + dropout in between
        self.projection = nn.Sequential(
            nn.Linear(hidden_size, 256),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(256, projection_dim)
        )

    def forward(self, input_values):
        # Extract per-frame hidden states (last transformer layer)
        outputs = self.wavlm(input_values=input_values)
        x = outputs.last_hidden_state

        # Mean-pool across time to obtain one vector per clip
        x = x.mean(dim=1)

        # Project to the contrastive embedding space and L2-normalise
        z = self.projection(x)
        z = F.normalize(z, dim=1)

        return z

model = WavLMSupConModel().to(DEVICE)
print("Model ready")

In [ ]:
# Layer-freezing strategy:
# Train all encoder layers EXCEPT the first 4. The lowest layers capture
# generic acoustic features that should remain stable; freezing them also
# reduces the number of trainable parameters and the risk of catastrophic
# forgetting on a small in-domain dataset.
for param in model.wavlm.parameters():
    param.requires_grad = True

# Freeze the first 4 transformer layers
for layer in model.wavlm.encoder.layers[:4]:
    for param in layer.parameters():
        param.requires_grad = False

# Always train the projection head
for param in model.projection.parameters():
    param.requires_grad = True

# Report the trainable parameter count for transparency
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())

print("Trainable params:", trainable)
print("Total params:", total)
print("Trainable ratio:", round(trainable / total * 100, 2), "%")

In [ ]:
# Supervised Contrastive Loss (Khosla et al., 2020).
# For every anchor in the batch, all samples sharing the anchor's label
# count as positives, and all other samples count as negatives.
class SupConLoss(nn.Module):
    def __init__(self, temperature=0.05):
        super().__init__()
        self.temperature = temperature

    def forward(self, features, labels):
        device = features.device
        batch_size = features.shape[0]

        # Build a binary mask: mask[i, j] = 1 if labels[i] == labels[j]
        labels = labels.view(-1, 1)
        mask = torch.eq(labels, labels.T).float().to(device)

        # All-pairs similarity, scaled by the temperature
        logits = torch.matmul(features, features.T) / self.temperature
        # Subtract the row-max for numerical stability of the log-softmax
        logits = logits - logits.max(dim=1, keepdim=True)[0].detach()

        # Remove self-similarity (diagonal) so an anchor is never its own positive
        logits_mask = torch.ones_like(mask) - torch.eye(batch_size, device=device)
        mask = mask * logits_mask

        # Log-softmax over all non-self entries, weighted by positive mask
        exp_logits = torch.exp(logits) * logits_mask
        log_prob = logits - torch.log(
            exp_logits.sum(dim=1, keepdim=True) + 1e-12
        )

        # Average over positives per anchor, then over the batch
        positives = mask.sum(dim=1)
        mean_log_prob_pos = (
            (mask * log_prob).sum(dim=1) / (positives + 1e-12)
        )

        loss = -mean_log_prob_pos.mean()
        return loss

# Tight temperature (0.05) emphasises hard negatives during training
criterion = SupConLoss(temperature=0.05)

# AdamW with conservative settings; lr is intentionally small for fine-tuning
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-5,
    weight_decay=1e-4,
    betas=(0.9, 0.98)
)

# Cosine annealing decays lr from 1e-5 down to 1e-6 over the full schedule
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=2000,
    eta_min=1e-6
)

print("Loss + optimizer + scheduler ready")

In [ ]:
# Persist checkpoints to Google Drive so they survive Colab session resets.
# Two helpers below: one for saving and one for resuming the latest step-tagged checkpoint.
PROJECT_DIR = "/content/drive/MyDrive/AbjadKids_WavLM_SupCon"
CKPT_DIR = os.path.join(PROJECT_DIR, "checkpoints")
FINAL_DIR = os.path.join(PROJECT_DIR, "final_model")

os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(FINAL_DIR, exist_ok=True)

print("CKPT_DIR:", CKPT_DIR)

def save_ckpt(model, optimizer, scheduler, step, name=None):
    # If `name` is provided, use it (e.g. for the best-val checkpoint);
    # otherwise fall back to a step-tagged filename.
    fname = name if name is not None else f"wavlm_supcon_step_{step}.pt"
    path = os.path.join(CKPT_DIR, fname)
    torch.save({
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "step": step,
        "label2id": label2id,
        "id2label": id2label,
    }, path)
    print(f"Saved: {path}")

def load_latest_ckpt(model, optimizer, scheduler):
    # Pick the *step-tagged* checkpoint with the highest step number
    files = [f for f in os.listdir(CKPT_DIR) if f.endswith(".pt") and "_step_" in f]

    if not files:
        print("No checkpoint found. Starting fresh.")
        return 0

    files = sorted(
        files,
        key=lambda x: int(x.split("_step_")[1].replace(".pt", ""))
    )

    latest = files[-1]
    path = os.path.join(CKPT_DIR, latest)

    ckpt = torch.load(path, map_location=DEVICE)

    model.load_state_dict(ckpt["model_state_dict"])
    optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    scheduler.load_state_dict(ckpt["scheduler_state_dict"])

    print(f"Resumed from: {path}")
    return ckpt["step"]

# Always start from step 0 here (set start_step via load_latest_ckpt to resume)
start_step = 0
print("Starting new optimizer schedule from step 0")

In [ ]:
# Validation evaluation function.
# The validation split is held out at the speaker level (see the splits cell)
# so any reduction in val SupCon loss reflects generalisation to unseen speakers.
VAL_BATCHES = 20  # number of mini-batches sampled to estimate validation loss

def evaluate_validation_loss(model, val_loader, criterion, device, num_batches=VAL_BATCHES):
    # Switch to eval mode so dropout/batchnorm behave deterministically
    model.eval()
    losses = []
    val_iter = iter(val_loader)
    with torch.no_grad():
        # Average the loss over a fixed number of validation batches
        for _ in range(num_batches):
            try:
                batch = next(val_iter)
            except StopIteration:
                break
            input_values = batch["input_values"].to(device, non_blocking=True)
            labels = batch["labels"].to(device, non_blocking=True)
            embeddings = model(input_values)
            loss = criterion(embeddings, labels)
            losses.append(loss.item())
    # Restore train mode for the next training step
    model.train()
    if not losses:
        return float("nan")
    return sum(losses) / len(losses)

print("Validation evaluation function ready")

In [ ]:
# Main training loop: SupCon fine-tuning with periodic validation.
TOTAL_STEPS = 2000
SAVE_EVERY = 100        # write a step-tagged checkpoint at this cadence
PRINT_EVERY = 10        # update the tqdm description every N steps
VALIDATE_EVERY = 100    # compute validation loss every N steps

model.train()
loader_iter = iter(train_loader)

pbar = tqdm(
    range(start_step, TOTAL_STEPS),
    initial=start_step,
    total=TOTAL_STEPS
)

best_val_loss = float("inf")   # tracks best validation loss seen so far
val_history = []               # (step, val_loss) tuples for the plot below

for step in pbar:
    # Pull the next balanced batch from the infinite sampler
    batch = next(loader_iter)

    input_values = batch["input_values"].to(DEVICE, non_blocking=True)
    labels = batch["labels"].to(DEVICE, non_blocking=True)

    # Standard forward / backward / step cycle
    optimizer.zero_grad()
    embeddings = model(input_values)
    loss = criterion(embeddings, labels)
    loss.backward()

    # Gradient clipping at L2-norm 1.0 keeps fine-tuning stable
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

    optimizer.step()
    scheduler.step()

    # Update the live progress bar with the current loss and learning rate
    if step % PRINT_EVERY == 0:
        lr = scheduler.get_last_lr()[0]
        pbar.set_description(
            f"Step {step} | Loss {loss.item():.4f} | LR {lr:.2e}"
        )

    # ─── Periodic validation ───
    if step > 0 and step % VALIDATE_EVERY == 0:
        val_loss = evaluate_validation_loss(model, val_loader, criterion, DEVICE)
        val_history.append((step, val_loss))
        pbar.write(f"Step {step} | Val SupCon Loss {val_loss:.4f}")

        # Save a separate "best validation" checkpoint when val loss improves
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            save_ckpt(model, optimizer, scheduler, step, name="wavlm_supcon_best.pt")
            pbar.write(f"  -> new best validation loss; checkpoint saved")

    # Periodic step-tagged checkpoint (independent of validation)
    if step > 0 and step % SAVE_EVERY == 0:
        save_ckpt(model, optimizer, scheduler, step)

# Always save a final checkpoint so the last-step model is recoverable
save_ckpt(model, optimizer, scheduler, TOTAL_STEPS)
print("Training finished")
print(f"Best validation SupCon loss: {best_val_loss:.4f}")

In [ ]:
# Plot the recorded validation loss curve.
# A monotonically decreasing curve indicates healthy fine-tuning;
# a curve that plateaus or rises is an early sign of overfitting.
import matplotlib.pyplot as plt

if val_history:
    steps_v, losses_v = zip(*val_history)
    plt.figure(figsize=(8, 4))
    plt.plot(steps_v, losses_v, marker="o")
    plt.xlabel("Training Step")
    plt.ylabel("Validation SupCon Loss")
    plt.title("Validation Loss During Fine-tuning")
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

In [ ]:
# Build a *small* evaluation set: at most 20 examples per class,
# drawn from the held-out test split. This keeps cosine-similarity
# and clustering metrics tractable while remaining representative.
model.eval()

test_eval_df = test_df.groupby("label", group_keys=False).apply(
    lambda x: x.sample(min(len(x), 20), random_state=42)
).reset_index(drop=True)

test_eval_dataset = AbjadAudioDataset(test_eval_df)

# Standard (un-balanced) loader is fine for inference
test_eval_loader = DataLoader(
    test_eval_dataset,
    batch_size=32,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=0
)

all_embs = []
all_labels = []

# Run the model in inference mode and collect projection embeddings + labels
with torch.no_grad():
    for batch in tqdm(test_eval_loader):
        input_values = batch["input_values"].to(DEVICE)
        labels = batch["labels"].cpu().numpy()

        embs = model(input_values).cpu().numpy()

        all_embs.append(embs)
        all_labels.extend(labels)

all_embs = np.vstack(all_embs)
all_labels = np.array(all_labels)

print("Embeddings:", all_embs.shape)
print("Labels:", all_labels.shape)
print("Classes:", len(set(all_labels)))

In [ ]:
# Top-1 / Top-5 retrieval accuracy on the held-out test embeddings.
# A query embedding is "correct" at rank k if any of the top-k nearest
# neighbours (excluding itself) shares its label.
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

sim = cosine_similarity(all_embs)

# Replace the self-similarity with -inf so the query never retrieves itself
np.fill_diagonal(sim, -999)

top1_correct = 0
top5_correct = 0

for i in range(len(all_labels)):
    # Sort other samples by descending similarity to the i-th query
    ranking = np.argsort(sim[i])[::-1]

    top1 = ranking[0]
    top5 = ranking[:5]

    if all_labels[top1] == all_labels[i]:
        top1_correct += 1

    if all_labels[i] in all_labels[top5]:
        top5_correct += 1

top1_acc = top1_correct / len(all_labels)
top5_acc = top5_correct / len(all_labels)

print("Top-1 Retrieval Accuracy:", round(top1_acc * 100, 2), "%")
print("Top-5 Retrieval Accuracy:", round(top5_acc * 100, 2), "%")

In [ ]:
# Same-class vs different-class similarity gap.
# A wide gap means same-word clips cluster tightly and different-word clips
# stay far apart, which is the geometric goal of contrastive training.
same_sims = []
diff_sims = []

for i in range(len(all_labels)):
    for j in range(i + 1, len(all_labels)):
        if all_labels[i] == all_labels[j]:
            same_sims.append(sim[i, j])
        else:
            diff_sims.append(sim[i, j])

same_sims = np.array(same_sims)
diff_sims = np.array(diff_sims)

print("Same-word similarity mean:", round(same_sims.mean(), 4))
print("Different-word similarity mean:", round(diff_sims.mean(), 4))
print("Gap:", round(same_sims.mean() - diff_sims.mean(), 4))

In [ ]:
# Agglomerative clustering with cosine distance, evaluated against the
# ground-truth labels using ARI, NMI, and Purity.
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
from sklearn.metrics.cluster import contingency_matrix
import numpy as np

# Purity: fraction of points whose cluster's most frequent class matches the point's class
def purity_score(y_true, y_pred):
    matrix = contingency_matrix(y_true, y_pred)
    return np.sum(np.amax(matrix, axis=0)) / np.sum(matrix)

# Average-linkage AHC with cosine distance over the projection embeddings
clustering = AgglomerativeClustering(
    n_clusters=len(set(all_labels)),
    metric="cosine",
    linkage="average"
)

pred_clusters = clustering.fit_predict(all_embs)

# Standard partition-quality metrics
ari = adjusted_rand_score(all_labels, pred_clusters)
nmi = normalized_mutual_info_score(all_labels, pred_clusters)
purity = purity_score(all_labels, pred_clusters)

print("ARI:", round(ari, 4))
print("NMI:", round(nmi, 4))
print("Purity:", round(purity, 4))

In [ ]:
# HDBSCAN baseline: density-based clustering that can leave outliers
# unassigned (label = -1). Noise points are excluded before computing metrics
# so the reported numbers reflect only the clustered subset.
import hdbscan
import numpy as np
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
from sklearn.metrics.cluster import contingency_matrix

def purity_score(y_true, y_pred):
    matrix = contingency_matrix(y_true, y_pred)
    return np.sum(np.amax(matrix, axis=0)) / np.sum(matrix)

# min_cluster_size=5 keeps small but meaningful clusters; min_samples=2 is
# the standard lower bound for stable density estimates.
clusterer = hdbscan.HDBSCAN(
    min_cluster_size=5,
    min_samples=2,
    metric='euclidean'
)

pred_clusters = clusterer.fit_predict(all_embs)

# Drop the noise points (-1) from the comparison
mask = pred_clusters != -1
y_true = all_labels[mask]
y_pred = pred_clusters[mask]

ari = adjusted_rand_score(y_true, y_pred)
nmi = normalized_mutual_info_score(y_true, y_pred)
purity = purity_score(y_true, y_pred)

print("Clusters found:", len(set(y_pred)))
print("Noise removed:", np.sum(pred_clusters == -1))
print("ARI:", round(ari, 4))
print("NMI:", round(nmi, 4))
print("Purity:", round(purity, 4))